# Plotly Full Correlation Batch Export

전기 대표 11개와 열 계량기 9개에 대해 Plotly full correlation heatmap을 생성하고 HTML로 저장합니다.

저장 경로:
- `outputs/electric_plotly_6year`
- `outputs/electric_plotly_yearly`
- `outputs/electric_plotly_seasonal`
- `outputs/thermal_plotly_6year`
- `outputs/thermal_plotly_yearly`
- `outputs/thermal_plotly_seasonal`

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.meter_metadata import get_meters_by_type
from scripts.eda_raw_correlation_representative_electric import (
    REPRESENTATIVE_METERS,
    SEASON_ORDER,
    add_period_columns,
    build_corr_matrix,
    build_engine,
    fetch_meter_raw_df as fetch_electric_meter_raw_df,
    fetch_weather_df,
    select_usable_columns,
)
from scripts.eda_raw_correlation_all_thermal import (
    fetch_meter_raw_df as fetch_thermal_meter_raw_df,
)

pio.renderers.default = 'notebook_connected'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
ELECTRIC_PLOTLY_6YEAR = OUTPUT_ROOT / 'electric_plotly_6year'
ELECTRIC_PLOTLY_YEARLY = OUTPUT_ROOT / 'electric_plotly_yearly'
ELECTRIC_PLOTLY_SEASONAL = OUTPUT_ROOT / 'electric_plotly_seasonal'
THERMAL_PLOTLY_6YEAR = OUTPUT_ROOT / 'thermal_plotly_6year'
THERMAL_PLOTLY_YEARLY = OUTPUT_ROOT / 'thermal_plotly_yearly'
THERMAL_PLOTLY_SEASONAL = OUTPUT_ROOT / 'thermal_plotly_seasonal'

for path in [
    ELECTRIC_PLOTLY_6YEAR,
    ELECTRIC_PLOTLY_YEARLY,
    ELECTRIC_PLOTLY_SEASONAL,
    THERMAL_PLOTLY_6YEAR,
    THERMAL_PLOTLY_YEARLY,
    THERMAL_PLOTLY_SEASONAL,
]:
    path.mkdir(parents=True, exist_ok=True)

THERMAL_METERS = get_meters_by_type('thermal')
engine = build_engine()
weather_df, weather_columns = fetch_weather_df(engine)

REPRESENTATIVE_METERS, THERMAL_METERS

In [ ]:
def build_plotly_corr_figure(corr_df: pd.DataFrame, title: str):
    columns = list(corr_df.columns)
    fig = go.Figure(
        data=go.Heatmap(
            z=corr_df.values,
            x=columns,
            y=columns,
            zmin=-1,
            zmax=1,
            colorscale='RdBu',
            reversescale=True,
            colorbar=dict(title='corr'),
            text=corr_df.round(2).values,
            texttemplate='%{text}',
            textfont=dict(size=9),
            hovertemplate='x=%{x}<br>y=%{y}<br>corr=%{z:.4f}<extra></extra>',
        )
    )
    fig.update_layout(
        title=title,
        width=max(900, len(columns) * 36),
        height=max(800, len(columns) * 32),
        xaxis=dict(tickangle=90),
        yaxis=dict(autorange='reversed'),
    )
    return fig


def prepare_meter_df(meter_urn: str, meter_kind: str):
    if meter_kind == 'electric':
        raw_df, measurements = fetch_electric_meter_raw_df(engine, meter_urn)
    else:
        raw_df, measurements = fetch_thermal_meter_raw_df(engine, meter_urn)
    merged_df = raw_df.merge(weather_df, on='ts', how='left')
    merged_df = add_period_columns(merged_df)
    candidate_columns = measurements + weather_columns
    usable_columns = select_usable_columns(merged_df, candidate_columns)
    return merged_df, measurements, usable_columns


def export_full_corr_html(df: pd.DataFrame, usable_columns: list[str], title: str, output_path: Path):
    corr_df = build_corr_matrix(df, usable_columns)
    fig = build_plotly_corr_figure(corr_df, title)
    fig.write_html(output_path, include_plotlyjs='cdn')
    return corr_df


In [ ]:
def export_electric_all():
    for meter_urn in REPRESENTATIVE_METERS:
        merged_df, measurements, usable_columns = prepare_meter_df(meter_urn, 'electric')
        export_full_corr_html(
            merged_df,
            usable_columns,
            f'{meter_urn} 6-Year Full Correlation Matrix',
            ELECTRIC_PLOTLY_6YEAR / f'{meter_urn}_6year_full_corr.html',
        )

        for year in sorted(int(year) for year in merged_df['year'].dropna().unique()):
            year_df = merged_df.loc[merged_df['year'] == year].copy()
            year_columns = select_usable_columns(year_df, measurements + weather_columns)
            if len(year_df) < 24 or len(year_columns) < 2:
                continue
            export_full_corr_html(
                year_df,
                year_columns,
                f'{meter_urn} {year} Full Correlation Matrix',
                ELECTRIC_PLOTLY_YEARLY / f'{meter_urn}_{year}_full_corr.html',
            )

        for season in SEASON_ORDER:
            season_df = merged_df.loc[merged_df['season'] == season].copy()
            season_columns = select_usable_columns(season_df, measurements + weather_columns)
            if len(season_df) < 24 or len(season_columns) < 2:
                continue
            export_full_corr_html(
                season_df,
                season_columns,
                f'{meter_urn} {season} Full Correlation Matrix',
                ELECTRIC_PLOTLY_SEASONAL / f'{meter_urn}_{season}_full_corr.html',
            )

    print('electric done')

In [ ]:
def export_thermal_all():
    for meter_urn in THERMAL_METERS:
        merged_df, measurements, usable_columns = prepare_meter_df(meter_urn, 'thermal')
        export_full_corr_html(
            merged_df,
            usable_columns,
            f'{meter_urn} 6-Year Full Thermal Correlation Matrix',
            THERMAL_PLOTLY_6YEAR / f'{meter_urn}_6year_full_corr.html',
        )

        for year in sorted(int(year) for year in merged_df['year'].dropna().unique()):
            year_df = merged_df.loc[merged_df['year'] == year].copy()
            year_columns = select_usable_columns(year_df, measurements + weather_columns)
            if len(year_df) < 24 or len(year_columns) < 2:
                continue
            export_full_corr_html(
                year_df,
                year_columns,
                f'{meter_urn} {year} Full Thermal Correlation Matrix',
                THERMAL_PLOTLY_YEARLY / f'{meter_urn}_{year}_full_corr.html',
            )

        for season in SEASON_ORDER:
            season_df = merged_df.loc[merged_df['season'] == season].copy()
            season_columns = select_usable_columns(season_df, measurements + weather_columns)
            if len(season_df) < 24 or len(season_columns) < 2:
                continue
            export_full_corr_html(
                season_df,
                season_columns,
                f'{meter_urn} {season} Full Thermal Correlation Matrix',
                THERMAL_PLOTLY_SEASONAL / f'{meter_urn}_{season}_full_corr.html',
            )

    print('thermal done')

## Run Batch Export

In [ ]:
export_electric_all()
export_thermal_all()

In [ ]:
sorted(str(path.relative_to(PROJECT_ROOT)) for path in ELECTRIC_PLOTLY_6YEAR.glob('*.html'))[:5], sorted(str(path.relative_to(PROJECT_ROOT)) for path in THERMAL_PLOTLY_6YEAR.glob('*.html'))[:5]